# 01 — Data Preprocessing
**Alzheimer's Disease Classification | KLE Technological University**

This notebook merges the ADNI cognitive dataset (`ADNIMERGE.csv`) with FreeSurfer 7.x MRI volumetric data (`UCSFFSX7_*.csv`) into a single analysis-ready CSV.

### Requirements
- ADNI data access approved at https://adni.loni.usc.edu
- Both CSV files downloaded and uploaded to Google Drive
- See `data/README.md` for full instructions

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# ---- Edit these paths to match your Drive ----
ADNIMERGE_PATH = '/content/drive/MyDrive/AD1/ADNIMERGE_10Oct2025.csv'
UCSF_FS_PATH   = '/content/drive/MyDrive/AD1/UCSFFSX7_28Oct2025.csv'
OUTPUT_PATH    = '/content/drive/MyDrive/AD1/ADNI_MRI_Cognitive_Merged.csv'

In [ ]:
# Load both datasets
adni = pd.read_csv(ADNIMERGE_PATH, low_memory=False)
mri  = pd.read_csv(UCSF_FS_PATH,   low_memory=False)

print('ADNIMERGE shape :', adni.shape)
print('FreeSurfer shape:', mri.shape)
print('\nADNIMERGE columns (first 10):', adni.columns[:10].tolist())
print('FreeSurfer columns (first 10):', mri.columns[:10].tolist())

In [ ]:
# Standardize visit code columns
for df in [adni, mri]:
    if 'VISCODE' in df.columns:
        df['VISCODE'] = df['VISCODE'].str.lower().str.strip()
    if 'VISCODE2' in df.columns:
        df['VISCODE2'] = df['VISCODE2'].str.lower().str.strip()

print('Visit codes standardized.')

In [ ]:
# Merge on subject ID and visit code
merged = pd.merge(
    adni, mri,
    how='inner',
    left_on=['RID', 'VISCODE'],
    right_on=['RID', 'VISCODE'],
    suffixes=('_cog', '_mri')
)

print('Merged shape:', merged.shape)

In [ ]:
# Select relevant columns
keep_cols = [
    'RID', 'VISCODE', 'DX_bl', 'AGE', 'PTGENDER', 'PTEDUCAT',
    'MMSE', 'ADAS13', 'CDRSB',
    'Hippocampus', 'Entorhinal'
]

# Drop APOE4 — model uses only MRI + cognitive features
keep_cols = [c for c in keep_cols if c in merged.columns]
final_df  = merged[keep_cols].dropna()

print('Final dataset shape:', final_df.shape)
print('\nClass distribution:')
print(final_df['DX_bl'].value_counts())
final_df.head()

In [ ]:
# Save merged dataset
final_df.to_csv(OUTPUT_PATH, index=False)
print(f'Merged dataset saved to: {OUTPUT_PATH}')